# Lesson: Bridging NetDevOps and AI


### What you'll do

**1. Hands-on "The NetDevOps and AI tooling landscape"**
- The `ask_network_ai()` helper — the glue layer reused across the remaining lessons

**2. Hands-on "Architectural patterns for AI-NetDevOps"**
- Pre-commit validation (Pattern 1) — catch a bad ACL push
- Closed-loop remediation (Pattern 2) — auto-generate the fix
- Choosing your pattern — match workflows to patterns

### Requirements
- Python 3.10+
- A local [Ollama](https://ollama.com) server at `http://127.0.0.1:11434` with `gemma4:e4b` pulled (`ollama pull gemma4:e4b`); no GPU required

### Running scenario
**IntentNet Corp** — continued from the previous lesson. The team has seen AI parse BGP output and generate ACLs; now they decide *where else* to deploy AI and how to wire it into their pipelines.

---

In [11]:
# ============================================================
# Environment Setup
# ============================================================
!pip install -q ollama

import json
import ollama

# Configure Ollama client to point at the local server
OLLAMA_HOST = "http://127.0.0.1:11434"
MODEL = "gemma4:e4b"

client = ollama.Client(host=OLLAMA_HOST)

print(f"Environment ready. Using {MODEL} at {OLLAMA_HOST}")

Environment ready. Using gemma4:e4b at http://127.0.0.1:11434


---
## 1. Hands-on "The NetDevOps and AI tooling landscape"

IntentNet already runs Ansible, NetBox, GitLab CI, and CML. Where does AI fit? Everywhere the stack produces or consumes text — and every integration runs through the same glue layer: one clean function around the LLM. That function is this hands-on's deliverable: `ask_network_ai()`, the helper you'll reuse for the rest of the course.

> You don't replace your NetDevOps stack. You wire it with AI.

The glue layer in one function: the Ollama client, the network-engineer system prompt, a `json_output` switch, and temperature pinned at 0. Every LLM call for the rest of the course goes through it.

In [13]:
# ============================================================
# Reusable LLM Helper Function
# ============================================================
# Wrap the Ollama client into a clean helper for the rest of the course.
# This is the "glue layer" pattern from the topic "The NetDevOps and AI tooling landscape".

NETWORK_SYSTEM_PROMPT = (
    "You are a senior Cisco network engineer with 15 years of experience.\n"
    "You work at IntentNet Corp managing 600 IOS-XE switches across 3 campus locations.\n"
    "When diagnosing issues:\n"
    "- Reference specific Cisco CLI commands\n"
    "- Consider both physical and logical causes\n"
    "- Suggest verification steps before remediation\n"
    "- Flag anything that could cause an outage if done incorrectly\n"
    "Keep responses concise and actionable."
)

def ask_network_ai(
    prompt: str,
    system_prompt: str = NETWORK_SYSTEM_PROMPT,
    json_output: bool = False,
    temperature: float = 0,
) -> str | dict:
    """Send a prompt to the LLM with network engineering context.

    Args:
        prompt: The user prompt / question.
        system_prompt: System prompt (defaults to NETWORK_SYSTEM_PROMPT).
        json_output: If True, request JSON response and parse it.
        temperature: Sampling temperature (0 = deterministic).

    Returns:
        Parsed dict if json_output=True, otherwise the raw string response.
    """
    kwargs = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "options": {"temperature": temperature},
    }
    if json_output:
        kwargs["format"] = "json"

    response = client.chat(**kwargs)
    content = response["message"]["content"]

    if json_output:
        return json.loads(content)
    return content


# Quick test — one structured question through the helper
test_result = ask_network_ai(
    "What Cisco CLI command shows OSPF neighbor adjacencies? "
    "Return JSON with: command, description, and example_output (3 lines).",
    json_output=True,
)

print("Helper function test:")
print(json.dumps(test_result, indent=2))
print("\nThe ask_network_ai() helper will be reused in the remaining lessons.")

Helper function test:
{
  "command": "show ip ospf neighbor",
  "description": "Displays the state of all configured OSPF neighbors on the local router. This is the primary command for verifying adjacency status.",
  "example_output": "Neighbor ID           Pri   State           Dead Time   Address\n10.1.1.2             1       FULL            00:00:35    192.168.1.2\n10.1.1.3             1       FULL            00:00:35    192.168.1.3"
}

The ask_network_ai() helper will be reused in the remaining lessons.


One clean function call per LLM interaction — model, system prompt, and output format in one place. When you switch models or hosts, you change one line, not every cell.

---
## 2. Hands-on "Architectural patterns for AI-NetDevOps"

Prototype the two most immediately useful of the four architectural patterns, then pick the right one for any workflow — three parts:

1. **Pre-commit validation (Pattern 1)** — catch a bad ACL push
2. **Closed-loop remediation (Pattern 2)** — generate the fix
3. **Choosing your pattern** — match workflows to patterns

**Scenario:** a junior engineer pushes an ACL change to the guest network with three policy violations. Can the AI catch them before production?

### Part 1 — Pre-Commit Validation (Pattern 1)

The LLM acts as a pre-commit hook: it reviews the diff against IntentNet's security policy and returns a verdict with the offending lines. Watch for the three violations.

In [16]:
# ============================================================
# Pattern 1: Pre-Commit Config Validation
# ============================================================
# Simulate: developer pushes a config change → LLM reviews → pass/fail

security_policy = """
IntentNet Corp — Network Security Policy (ACL Standards)
=========================================================
1. SSH (port 22) access from the guest network (10.0.0.0/8) MUST be denied to all destinations.
2. The guest network (10.0.0.0/8) MUST NOT have unrestricted IP access to any network.
3. All extended ACLs MUST end with an explicit "deny ip any any log" statement.
4. HTTPS (port 443) from guest to the server network (172.16.0.0/16) is permitted.
5. DNS (port 53, UDP) from guest to the DNS servers (192.168.1.10, 192.168.1.11) is permitted.
"""

config_diff = """
--- a/acl/guest-restrict.cfg
+++ b/acl/guest-restrict.cfg
@@ -1,4 +1,6 @@
 ip access-list extended GUEST-RESTRICT
+  permit tcp 10.0.0.0 0.255.255.255 172.16.0.0 0.0.255.255 eq 443
+  permit tcp 10.0.0.0 0.255.255.255 172.16.0.0 0.0.255.255 eq 22
+  permit udp 10.0.0.0 0.255.255.255 host 192.168.1.10 eq 53
+  permit udp 10.0.0.0 0.255.255.255 host 192.168.1.11 eq 53
+  permit ip 10.0.0.0 0.255.255.255 any
"""

validation_prompt = (
    "You are a pre-commit validation hook in IntentNet Corp's CI/CD pipeline.\n"
    "Review the following config diff against the security policy.\n\n"
    "Return JSON with:\n"
    "- verdict: PASS or FAIL (string)\n"
    "- violations: array of objects, each with line (the offending config line), "
    "policy_rule (which rule it violates), and explanation (why it's a violation)\n"
    "- summary: one-sentence summary of the review result (string)\n\n"
    f"SECURITY POLICY:\n{security_policy}\n\n"
    f"CONFIG DIFF:\n{config_diff}"
)

validation_result = ask_network_ai(validation_prompt, json_output=True)

print("Pre-Commit Config Validation Result")
print("=" * 50)
print(f"Verdict: {validation_result.get('verdict', 'UNKNOWN')}")
print(f"Summary: {validation_result.get('summary', '')}")
print()

violations = validation_result.get("violations", [])
for i, v in enumerate(violations, 1):
    print(f"Violation #{i}:")
    print(f"  Line:        {v.get('line', '')}")
    print(f"  Policy Rule: {v.get('policy_rule', '')}")
    print(f"  Explanation: {v.get('explanation', '')}")
    print()

Pre-Commit Config Validation Result
Verdict: FAIL
Summary: The configuration fails validation due to permitting SSH access, allowing unrestricted IP traffic, and missing the mandatory final logging deny statement.

Violation #1:
  Line:        permit tcp 10.0.0.0 0.255.255.255 172.16.0.0 0.0.255.255 eq 22
  Policy Rule: 1. SSH (port 22) access from the guest network (10.0.0.0/8) MUST be denied to all destinations.
  Explanation: This line explicitly permits SSH traffic (port 22) from the guest network (10.0.0.0/8) to the server network (172.16.0.0/16), violating the policy that SSH access must be denied.

Violation #2:
  Line:        permit ip 10.0.0.0 0.255.255.255 any
  Policy Rule: 2. The guest network (10.0.0.0/8) MUST NOT have unrestricted IP access to any network.
  Explanation: This line permits all IP traffic from the guest network (10.0.0.0/8) to any destination ('any'), which constitutes unrestricted access and violates the security policy.

Violation #3:
  Line:        permi

**Verdict: FAIL** — the model caught all three violations: SSH permitted from the guest network, the unrestricted `permit ip ... any`, and the missing final `deny ip any any log`. A human reviewer skims a 50-switch ACL change; the LLM doesn't.

---
### Part 2 — Closed-Loop Remediation (Pattern 2)

Now feed the violations back and ask for the corrected config — remove the violations, keep the legitimate permits. The human still approves: that's non-negotiable.

In [17]:
# ============================================================
# Pattern 2: Generate Remediation from Violations
# ============================================================
# Take the violations from Pattern 1 and generate a corrected config.
# This is the second step in a closed-loop remediation pattern.

# Build a summary of violations for the remediation prompt
violation_summary = "\n".join(
    f"- {v.get('line', '')}: {v.get('explanation', '')}"
    for v in violations
)

remediation_prompt = (
    "You are a remediation engine in IntentNet Corp's CI/CD pipeline.\n"
    "A config diff was reviewed and the following violations were found:\n\n"
    f"{violation_summary}\n\n"
    f"The original config diff was:\n{config_diff}\n\n"
    f"The security policy is:\n{security_policy}\n\n"
    "Generate a corrected version of the config that:\n"
    "1. Removes or fixes all violations\n"
    "2. Preserves all legitimate permit statements\n"
    "3. Complies fully with the security policy\n\n"
    "Return JSON with:\n"
    "- corrected_config: the full corrected ACL (string, IOS-XE syntax)\n"
    "- changes_made: array of strings describing each fix\n"
    "- review_notes: anything a human reviewer should double-check (string)\n"
)

remediation_result = ask_network_ai(remediation_prompt, json_output=True)

print("Remediation — Corrected Config")
print("=" * 50)
print(remediation_result.get("corrected_config", ""))
print()

print("Changes Made:")
for change in remediation_result.get("changes_made", []):
    print(f"  - {change}")
print()

print("Review Notes (for human approver):")
print(f"  {remediation_result.get('review_notes', '')}")
print()
print("--- Pipeline Flow ---")
print("  git push --> AI validation (FAIL) --> AI remediation --> human review --> merge")
print("  The AI caught the violations AND generated a fix. A human still approves.")

Remediation — Corrected Config
ip access-list extended GUEST-RESTRICT
 permit tcp 10.0.0.0 0.255.255.255 172.16.0.0 0.0.255.255 eq 443
 permit udp 10.0.0.0 0.255.255.255 host 192.168.1.10 eq 53
 permit udp 10.0.0.0 0.255.255.255 host 192.168.1.11 eq 53
 deny ip any any log

Changes Made:
  - Removed the explicit 'permit tcp ... eq 22' statement to comply with Policy Rule 1 (SSH must be denied).
  - Removed the overly permissive 'permit ip 10.0.0.0 0.255.255.255 any' statement to comply with Policy Rule 2 (No unrestricted access).
  - Added a final 'deny ip any any log' statement to ensure all remaining traffic is blocked and logged, complying with Policy Rule 3.

Review Notes (for human approver):
  Verify that the ACL name (GUEST-RESTRICT) is correctly applied to the ingress interface connecting the guest network segment. Since this is a security policy enforcement point, confirm if logging on the 'deny' statement is required for compliance or if only specific high-severity denies nee

The full loop: *git push → AI validation (FAIL) → AI remediation → human review → merge*. The AI caught it AND fixed it — and a human still owns the merge button.

---
### Part 3 — Choosing Your Pattern

Four patterns in the catalog. Describe a workflow — IntentNet's tedious Monday config-review session — and let the model recommend one, with reasoning and first steps.

In [21]:
# ============================================================
# Pattern Selector — Which Pattern Fits Your Workflow?
# ============================================================
# Describe a workflow and get a pattern recommendation.

pattern_catalog = """
Available AI-NetDevOps Architectural Patterns:

PATTERN 1: Pre-Commit Config Validation
  Trigger: Git push / merge request
  Flow: Config diff -> LLM review -> PASS/FAIL verdict
  Best for: Security policy compliance, syntax validation, best practice checks

PATTERN 2: Closed-Loop Remediation
  Trigger: Anomaly detected (monitoring, syslog, telemetry)
  Flow: Detect -> Diagnose -> Generate fix -> Human approval -> Apply
  Best for: Known failure modes, interface flaps, routing issues

PATTERN 3: RAG-Powered Knowledge Assistant
  Trigger: Engineer asks a question
  Flow: Query -> Search internal docs -> LLM generates grounded answer
  Best for: Tribal knowledge capture, onboarding, design standard lookups

PATTERN 4: Agentic Workflow
  Trigger: Complex multi-step task
  Flow: AI agent plans steps -> Executes tools -> Human approval gates -> Report
  Best for: Incident response, change management, capacity planning
"""

workflow_description = (
    "Our team spends 3 hours every Monday reviewing config changes from the "
    "previous week against our security baseline. We check for NTP authentication, "
    "SNMPv3 enforcement, and unused interface shutdown compliance. It's tedious "
    "and we sometimes miss things."
)

selector_prompt = (
    f"{pattern_catalog}\n\n"
    f"Workflow description: {workflow_description}\n\n"
    "Based on this workflow, recommend the best pattern. Return JSON with:\n"
    "- recommended_pattern: pattern number and name (string)\n"
    "- reasoning: why this pattern fits (string)\n"
    "- implementation_steps: 3-4 concrete next steps to get started (array of strings)\n"
    "- estimated_time_savings: how much time this could save per week (string)\n"
)

recommendation = ask_network_ai(selector_prompt, json_output=True)

print("Pattern Recommendation")
print("=" * 50)
print(f"Pattern:      {recommendation.get('recommended_pattern', '')}")
print(f"Reasoning:    {recommendation.get('reasoning', '')}")
print(f"Time Savings: {recommendation.get('estimated_time_savings', '')}")
print()
print("Implementation Steps:")
for step in recommendation.get("implementation_steps", []):
    print(f"  - {step}")

Pattern Recommendation
Pattern:      PATTERN 1: Pre-Commit Config Validation
Reasoning:    The current workflow is a manual, post-facto compliance audit of configuration changes. Pattern 1 directly addresses this by shifting the validation process 'left'—from a weekly human review to an automated check *before* the config hits the device (Git push/merge request). This ensures that non-compliant configurations are rejected immediately, preventing them from ever being deployed.
Time Savings: 2.5 - 3 hours per week

Implementation Steps:
  - Integrate a Git pre-commit hook or CI/CD pipeline step that triggers a configuration diff comparison against the established security baseline (e.g., NTP auth, SNMPv3).
  - Develop custom validation scripts (Python/Ansible) that parse the config diff and check for specific keywords or patterns (e.g., `ntp authenticate`, `snmp-server community v3`).
  - Use an LLM layer to interpret the failure report from the script, providing human-readable explanati

---
## Lesson Summary

You:

1. **Built `ask_network_ai()`** — the glue-layer helper reused across the remaining lessons
2. **Prototyped Pattern 1** — pre-commit config validation that caught the policy violations
3. **Prototyped Pattern 2** — closed-loop remediation that generated the corrected config
4. **Matched patterns to workflows** — a selector that recommends the right pattern for a described workflow

**Key insight:** the value of AI in NetDevOps is knowing *which problems to aim it at* — and the architectural patterns are the proven blueprints for wiring it in.